In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from moe_model import Expert,MoEConfig


In [10]:
config = MoEConfig()
experts = nn.ModuleList([Expert(config) for _ in range(config.n_experts)])

In [25]:
B = 1
CL = 4

T = B*CL

n_exp = 5
top_k = 2

ED = 8
x = torch.randn(T, ED)     
print("x shape ", x.shape)
gate = nn.Linear(ED, n_exp)
logits = gate(x)
print("logits shape", logits.shape)

probs  = F.softmax(logits, dim=-1)  # (T, N_experts)  each row sums to 1

top_k_probs, top_k_idx = torch.topk(probs, top_k, dim=-1)
print("top_k_probs shape", top_k_probs.shape)
print("top_k_idx shape", top_k_idx.shape)
print("top_k_idx", top_k_idx)

x shape  torch.Size([4, 8])
logits shape torch.Size([4, 5])
top_k_probs shape torch.Size([4, 2])
top_k_idx shape torch.Size([4, 2])
top_k_idx tensor([[3, 1],
        [3, 1],
        [2, 4],
        [3, 2]])


In [24]:
for i, expert in enumerate(experts):
    token_mask_2d = (top_k_idx == i)
    print(token_mask_2d)
    routed        = token_mask_2d.any(dim=-1) 
    print(routed)
    print(x[routed].size())
    break

tensor([[False,  True],
        [False,  True],
        [ True, False],
        [False, False]])
tensor([ True,  True,  True, False])
torch.Size([3, 8])
